# 🔧 Feature Engineering Masterclass

> **Master advanced feature engineering techniques for superior ML performance**

This notebook covers comprehensive feature engineering techniques that can significantly improve your machine learning models.

## 🎯 Learning Objectives

By the end of this notebook, you will:
- **Master** advanced feature creation techniques
- **Understand** feature selection methods
- **Handle** categorical variables effectively
- **Create** domain-specific features
- **Build** automated feature engineering pipelines

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import *
from sklearn.feature_selection import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("✅ All imports successful!")

## 🔧 Advanced Feature Creation

In [ ]:
class AdvancedFeatureEngineer:
    def __init__(self):
        self.feature_names = []
        self.transformers = {}
    
    def create_polynomial_features(self, df, columns, degree=2):
        """Create polynomial features"""
        new_features = df.copy()
        
        for col in columns:
            for d in range(2, degree + 1):
                new_col = f"{col}_poly_{d}"
                new_features[new_col] = df[col] ** d
                self.feature_names.append(new_col)
        
        return new_features
    
    def create_interaction_features(self, df, columns):
        """Create interaction features between column pairs"""
        new_features = df.copy()
        
        for i, col1 in enumerate(columns):
            for col2 in columns[i+1:]:
                new_col = f"{col1}_x_{col2}"
                new_features[new_col] = df[col1] * df[col2]
                self.feature_names.append(new_col)
        
        return new_features
    
    def create_ratio_features(self, df, numerator_cols, denominator_cols):
        """Create ratio features"""
        new_features = df.copy()
        
        for num_col in numerator_cols:
            for den_col in denominator_cols:
                if num_col != den_col:
                    new_col = f"{num_col}_div_{den_col}"
                    new_features[new_col] = df[num_col] / (df[den_col] + 1e-8)
                    self.feature_names.append(new_col)
        
        return new_features
    
    def create_binned_features(self, df, columns, n_bins=5):
        """Create binned categorical features from continuous variables"""
        new_features = df.copy()
        
        for col in columns:
            new_col = f"{col}_binned"
            new_features[new_col] = pd.cut(df[col], bins=n_bins, labels=False)
            self.feature_names.append(new_col)
        
        return new_features
    
    def create_statistical_features(self, df, columns, window=None):
        """Create statistical features (rolling if window specified)"""
        new_features = df.copy()
        
        for col in columns:
            if window:
                # Rolling statistics
                new_features[f"{col}_rolling_mean_{window}"] = df[col].rolling(window).mean()
                new_features[f"{col}_rolling_std_{window}"] = df[col].rolling(window).std()
            else:
                # Global statistics
                new_features[f"{col}_zscore"] = (df[col] - df[col].mean()) / df[col].std()
                new_features[f"{col}_percentile"] = df[col].rank(pct=True)
        
        return new_features

print("✅ AdvancedFeatureEngineer class defined!")

In [ ]:
# Create sample dataset
np.random.seed(42)
n_samples = 1000

data = {
    'age': np.random.normal(35, 10, n_samples),
    'income': np.random.lognormal(10, 0.5, n_samples),
    'experience': np.random.exponential(5, n_samples),
    'education_years': np.random.normal(16, 3, n_samples),
    'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], n_samples),
    'department': np.random.choice(['Engineering', 'Sales', 'Marketing', 'HR'], n_samples)
}

df = pd.DataFrame(data)
df['target'] = (df['income'] > df['income'].median()).astype(int)

print(f"Original dataset shape: {df.shape}")
df.head()

In [ ]:
# Apply feature engineering
fe = AdvancedFeatureEngineer()

# Select numeric columns for feature engineering
numeric_cols = ['age', 'income', 'experience', 'education_years']

# Create polynomial features
df_poly = fe.create_polynomial_features(df, ['age', 'experience'], degree=2)

# Create interaction features
df_interact = fe.create_interaction_features(df_poly, numeric_cols)

# Create ratio features
df_ratio = fe.create_ratio_features(df_interact, ['income'], ['age', 'experience', 'education_years'])

# Create binned features
df_binned = fe.create_binned_features(df_ratio, numeric_cols, n_bins=5)

print(f"Enhanced dataset shape: {df_binned.shape}")
print(f"New features created: {len(fe.feature_names)}")
print(f"Sample new features: {fe.feature_names[:5]}")

## 🎯 Feature Selection Techniques

In [ ]:
class FeatureSelector:
    def __init__(self):
        self.selected_features = []
        self.feature_scores = {}
    
    def univariate_selection(self, X, y, k=10):
        """Select k best features using univariate statistical tests"""
        selector = SelectKBest(score_func=f_classif, k=k)
        X_selected = selector.fit_transform(X, y)
        
        selected_indices = selector.get_support(indices=True)
        self.selected_features = X.columns[selected_indices].tolist()
        self.feature_scores['univariate'] = dict(zip(X.columns, selector.scores_))
        
        return X_selected, self.selected_features
    
    def recursive_feature_elimination(self, X, y, estimator, n_features=10):
        """Recursive feature elimination"""
        rfe = RFE(estimator=estimator, n_features_to_select=n_features)
        X_selected = rfe.fit_transform(X, y)
        
        selected_indices = rfe.get_support(indices=True)
        self.selected_features = X.columns[selected_indices].tolist()
        self.feature_scores['rfe'] = dict(zip(X.columns, rfe.ranking_))
        
        return X_selected, self.selected_features
    
    def feature_importance_selection(self, X, y, estimator, threshold=0.01):
        """Select features based on importance from tree-based models"""
        estimator.fit(X, y)
        importances = estimator.feature_importances_
        
        selected_indices = importances > threshold
        self.selected_features = X.columns[selected_indices].tolist()
        self.feature_scores['importance'] = dict(zip(X.columns, importances))
        
        return X.iloc[:, selected_indices], self.selected_features
    
    def plot_feature_importance(self, method='importance', top_n=20):
        """Plot feature importance scores"""
        if method not in self.feature_scores:
            print(f"No scores available for method: {method}")
            return
        
        scores = self.feature_scores[method]
        
        # Sort by score
        if method == 'rfe':
            # For RFE, lower rank is better
            sorted_features = sorted(scores.items(), key=lambda x: x[1])
        else:
            # For others, higher score is better
            sorted_features = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        
        # Take top N
        top_features = sorted_features[:top_n]
        
        features, values = zip(*top_features)
        
        plt.figure(figsize=(12, 8))
        plt.barh(range(len(features)), values)
        plt.yticks(range(len(features)), features)
        plt.xlabel('Score')
        plt.title(f'Top {top_n} Features - {method.title()} Method')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

print("✅ FeatureSelector class defined!")

In [ ]:
# Prepare data for feature selection
# Encode categorical variables
df_encoded = pd.get_dummies(df_binned, columns=['city', 'department'])

# Separate features and target
X = df_encoded.drop('target', axis=1)
y = df_encoded['target']

print(f"Total features before selection: {X.shape[1]}")

# Apply feature selection
selector = FeatureSelector()

# Method 1: Univariate selection
X_univariate, features_univariate = selector.univariate_selection(X, y, k=15)
print(f"Features selected by univariate method: {len(features_univariate)}")

# Method 2: Feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
X_importance, features_importance = selector.feature_importance_selection(X, y, rf, threshold=0.01)
print(f"Features selected by importance method: {len(features_importance)}")

# Plot feature importance
selector.plot_feature_importance('importance', top_n=15)

## 🎯 Practice Problems

### **Problem 1: Domain-Specific Feature Engineering**
Create features specific to a business domain.

In [ ]:
def create_business_features(df):
    """
    Create business-specific features for employee data
    
    Features to create:
    - Experience to age ratio
    - Income per year of education
    - Seniority level based on experience
    - Department salary comparison
    
    Returns:
    pd.DataFrame: DataFrame with new business features
    """
    # Your code here
    pass

# Test your function
# df_business = create_business_features(df)

### **Problem 2: Automated Feature Engineering Pipeline**
Create an automated pipeline that applies multiple feature engineering techniques.

In [ ]:
class AutoFeatureEngineer:
    def __init__(self, max_features=100):
        self.max_features = max_features
        self.feature_pipeline = []
    
    def fit_transform(self, X, y):
        """
        Automatically apply feature engineering and selection
        
        Steps:
        1. Create polynomial features
        2. Create interaction features
        3. Create statistical features
        4. Select best features
        5. Return transformed dataset
        
        Returns:
        pd.DataFrame: Transformed feature matrix
        """
        # Your code here
        pass

# Test your pipeline
# auto_fe = AutoFeatureEngineer(max_features=50)
# X_auto = auto_fe.fit_transform(X, y)

## 🎯 Key Takeaways

1. **Feature engineering** often provides the biggest performance gains
2. **Domain knowledge** is crucial for creating meaningful features
3. **Feature selection** prevents overfitting and improves interpretability
4. **Automated approaches** can discover unexpected patterns
5. **Validation** is essential to avoid data leakage

## 🔗 Next Steps

1. **Complete the practice problems** above
2. **Apply to domain-specific datasets**
3. **Move to the next notebook**: Ensemble Methods

---

**Outstanding feature engineering skills!** 🎉 You can now create powerful features for any ML problem.